# Bronze Layer Data Ingestion

Ingests raw CSV files from the `puc_data_specialist_de_2026_09.00-raw.brazil-ecommerce` volume into Delta Lake tables in the `01-bronze` schema. Each source file is converted to a typed Delta table with an `_ingested_at` metadata column for auditability.

Many of the code used in this Notebook has been provided by Databricks Labs training available here: https://github.com/IvanJPC/databricks-labs-ago-2026-data-ingestion/

The focus is use SQL statements instead of Apache Spark. Therefore, this notebook uses Apache Spark when executing SQL statements. It enables the power of using Python logical code with SQL. 

A improvement for this work is to change some of the SQL statements to use Spark only like Auto Loader or SQL Copy Into statement

In [0]:
%sql
--Setting UP Notebook variables and useful SQL commands to reduce typos
USE CATALOG puc_data_specialist_de_2026_09;
USE SCHEMA `01-bronze`;

In [0]:
from datetime import datetime

dt = datetime.now().strftime("%Y-%m-%d")

base_path = f"/Volumes/puc_data_specialist_de_2026_09/00-raw/brazil-ecommerce/{dt}"
print(base_path)

Understanding each file schema and data volume to create the SQL create table commands for each CSV 

In [0]:

csv_files = [
     "olist_customers_dataset.csv"

    # , "olist_orders_dataset.csv"
    # , "olist_geolocation_dataset.csv"
    # , "olist_order_items_dataset.csv"
    # , "olist_order_payments_dataset.csv"
    # , "olist_order_reviews_dataset.csv"
    # , "olist_products_dataset.csv"
    # , "olist_sellers_dataset.csv"
    # , "product_category_name_translation.csv"
]

for file_name in csv_files:
    df = spark.read.csv(f"{base_path}/{file_name}", header=True, inferSchema=True)
    print(f"=== {file_name} ===")
    print(f"Columns: {df.columns}")
    print(f"Row count: {df.count()}")
    df.show(5)
    print("---------------------------------------------")

    

In [0]:

for file_name in csv_files:

    bronze_table_name = file_name.removesuffix("_dataset.csv").removeprefix("olist_")
    print(f"Bronze table name to be created: {bronze_table_name}")
    
    ## Drop the table if it exists
    query = f"""DROP TABLE IF EXISTS {bronze_table_name};"""
    spark.sql(query)

    
    query = f""" CREATE TABLE {bronze_table_name} AS
      SELECT * 
     FROM read_files(
             '{base_path}/{file_name}',
             format => 'csv',
             schemaHints => 'customer_zip_code_prefix string'
         );"""
    spark.sql(query)
    
    query = f"""DESCRIBE TABLE EXTENDED {bronze_table_name};"""
    display(spark.sql(query))

